🧱 1. Imports

In [ ]:
# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
# from pypdf import PdfReader
import os
from transformers import pipeline
from tqdm.notebook import tqdm

/Users/kristinnuyens/Desktop/BeCode_Projects/CDM-RAG-Assistant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Initialize model

In [ ]:
generator = pipeline(
    "text-generation",
    model="NousResearch/Nous-Hermes-13b",
    device=-1,       # -1 = CPU, 0 = GPU
    do_sample=True,
    temperature=0.7,
    max_new_tokens=300
)

📄 2. Load PDFs with page numbers and headings (& other files code may still need to be added later)

In [2]:
# 🔴 ADAPT THIS PATH
folder_path = "../data/raw/"

# Simple heading detection heuristic
def extract_heading(text):
    lines = text.split("\n")
    for line in lines:
        if len(line.strip()) > 3 and line.strip().isupper():
            return line.strip()
    return "Unknown"

def load_pdfs_with_pages(folder_path):
    documents = []

    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        return documents

    # Walk recursively to include subfolders
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".pdf"):
                full_path = os.path.join(root, file)
                try:
                    reader = PdfReader(full_path)
                    for i, page in enumerate(reader.pages):
                        text = page.extract_text()
                        if text and text.strip():
                            heading = extract_heading(text)
                            documents.append({
                                "text": text.strip(),
                                "source": file,
                                "page": i+1,      # Pages are 1-indexed
                                "heading": heading
                            })
                        else:
                            print(f"⚠️ Empty page {i+1} in {file}")
                except Exception as e:
                    print(f"❌ Error reading {file}: {e}")

    return documents

# Load all PDFs
docs = load_pdfs_with_pages(folder_path)
print(f"\n✅ Loaded {len(docs)} pages from PDFs")


✅ Loaded 106 pages from PDFs


✂️ 3. Chunk the text

In [3]:
# Split text into chunks and keep page + heading
chunk_size = 500  # words

chunks = []
metadata = []

for doc in docs:
    words = doc["text"].split()
    for i in range(0, len(words), chunk_size):
        chunk_text = " ".join(words[i:i+chunk_size])
        chunks.append(chunk_text)
        metadata.append({
            "source": doc["source"],
            "page": doc["page"],
            "heading": doc["heading"]
        })

print(f"✅ Created {len(chunks)} chunks")

✅ Created 148 chunks


🧠 4. Create embeddings

In [4]:
from sentence_transformers import SentenceTransformer

# This will download the model from HuggingFace (faster on a stable connection)
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21896.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import numpy as np

# Generate embeddings
embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

print(f"✅ Embeddings created: {embeddings.shape}")

✅ Embeddings created: (148, 384)


🔎 5. Store in FAISS

In [6]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ FAISS index ready with {index.ntotal} vectors")

✅ FAISS index ready with 148 vectors


❓ 6. Ask a question (retrieval only)

In [7]:
def search(query, top_k=5, threshold=None):
    # 1️⃣ Encode the query
    query_embedding = model.encode([query]).astype("float32")

    # 2️⃣ Search top_k vectors
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for dist, idx in zip(distances[0], indices[0]):
        if threshold is None or dist < threshold:
            results.append({
                "text": chunks[idx],
                "source": metadata[idx]["source"],
                "page": metadata[idx]["page"],
                "heading": metadata[idx]["heading"],
                "distance": dist
            })
    return results

question = "data science skills"  # <-- Can update test question here
results = search(question, top_k=5)  # threshold optional

# 3️⃣ Display results with preview
for i, r in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source : {r['source']}")
    print(f"Page   : {r['page']}")
    print(f"Heading: {r['heading']}")
    print(f"Distance: {r['distance']:.4f}")
    print(f"Preview: {r['text'][:300]}")  # first 300 chars


--- Result 1 ---
Source : 2019_Evolution-of-CDM-to-CDS-Part-1-Drivers.pdf
Page   : 20
Heading: Unknown
Distance: 0.9759
Preview: The Evolution of Clinical Data Management into Clinical Data Science Society for Clinical Data Management Reflection Paper 20 b) Foundational Clinical Data Management Competencies While the role is evolving, several of the foundational competencies for a Clinical Data Manager remain the same. For ex

--- Result 2 ---
Source : 2020_Evolution-of-CDM-to-CDS-Part-3-Evolution-of-CDM-Role.pdf
Page   : 1
Heading: Unknown
Distance: 1.0530
Preview: The Evolution of Clinical Data Management into Clinical Data Science (Part 3: The evolution of the CDM role) A Reflection Paper on the evolution of CDM skillsets and competencies 31 Aug 2020 Society for Clinical Data Management © Society for Clinical Data Management Our Vision “Leading innovative cl

--- Result 3 ---
Source : SCDM-Position-Paper-Evolution-into-Clinical-to-Data-Science-V9.0.pdf
Page   : 7
Heading: Unknown
D

🤖 7. Add simple answer generation

In [9]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="NousResearch/Nous-Hermes-13b",
    device=-1  # CPU
)

Loading weights: 100%|██████████| 363/363 [00:00<00:00, 85010.18it/s]


In [11]:
from gpt4all import GPT4All

# ----------------------------
# 1️⃣ Initialize local GPT4All model
# ----------------------------
# Make sure you have downloaded the small model first (ggml-gpt4all-j-v1.bin or similar)
# Adapt path if needed
model_path = "./models/ggml-gpt4all-j-v1.bin"  # <-- adapt if your model is elsewhere
model = GPT4All(model_path)

# ----------------------------
# 2️⃣ Function to query LLM with top-k chunks
# ----------------------------
def answer_from_chunks(question, retrieved_chunks, max_tokens=300):
    """
    Args:
        question (str): The user query
        retrieved_chunks (list of dicts): Each dict has 'text', 'source', 'page', 'heading'
        max_tokens (int): Max tokens for LLM output
    Returns:
        str: Answer from LLM
    """
    # Combine text from top chunks
    context = "\n\n".join([
        f"Source: {c.get('source','Unknown')}, Page: {c.get('page','Unknown')}, Heading: {c.get('heading','Unknown')}\nText: {c['text']}" 
        for c in retrieved_chunks
    ])

    prompt = f"""
You are a Clinical Data Management assistant.

Answer the question below using ONLY the context provided. Cite PDF name, page, and heading when possible.

Context:
{context}

Question: {question}

Answer (concise, cite sources):
"""

    # Generate output from LLM
    output = model.generate(prompt, max_tokens=max_tokens)
    return output

# ----------------------------
# 3️⃣ Example Usage
# ----------------------------
# Suppose you already retrieved top-k chunks from FAISS
# Example dummy structure:
retrieved_chunks = [
    {'text': 'CDM roles evolve into CDS roles...', 'source': 'SCDM-Position-Paper.pdf', 'page': 5, 'heading': 'Evolution Overview'},
    {'text': 'Clinical Data Scientists need knowledge in SAS, R...', 'source': '2020_Evolution-of-CDM.pdf', 'page': 8, 'heading': 'Skills Needed'}
]

question = "How should a Clinical Data Manager transition to Data Science roles?"
answer = answer_from_chunks(question, retrieved_chunks)
print("----- ANSWER -----")
print(answer)

ValueError: Request failed: HTTP 404 Not Found

In [12]:
from transformers import pipeline

# ----------------------------
# 1️⃣ Initialize Nous Hermes model
# ----------------------------
generator = pipeline(
    "text-generation",
    model="NousResearch/Nous-Hermes-13b",
    device=-1,       # -1 for CPU, or 0 for GPU
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7
)

# ----------------------------
# 2️⃣ Function to query LLM with top-k chunks
# ----------------------------
def answer_from_chunks(question, retrieved_chunks):
    """
    Args:
        question (str): The user query
        retrieved_chunks (list of dicts): Each dict has 'text', 'source', 'page', 'heading'
    Returns:
        str: Answer from LLM
    """
    # Combine text from top chunks
    context = "\n\n".join([
        f"Source: {c.get('source','Unknown')}, Page: {c.get('page','Unknown')}, Heading: {c.get('heading','Unknown')}\nText: {c['text']}" 
        for c in retrieved_chunks
    ])

    prompt = f"""
You are a Clinical Data Management assistant.

Answer the question below using ONLY the context provided. Cite PDF name, page, and heading when possible.

Context:
{context}

Question: {question}

Answer (concise, cite sources):
"""

    # Generate output from Hugging Face model
    output = generator(prompt, max_new_tokens=300)[0]['generated_text']
    return output

# ----------------------------
# 3️⃣ Example Usage
# ----------------------------
retrieved_chunks = [
    {'text': 'CDM roles evolve into CDS roles...', 'source': 'SCDM-Position-Paper.pdf', 'page': 5, 'heading': 'Evolution Overview'},
    {'text': 'Clinical Data Scientists need knowledge in SAS, R...', 'source': '2020_Evolution-of-CDM.pdf', 'page': 8, 'heading': 'Skills Needed'}
]

question = "How should a Clinical Data Manager transition to Data Science roles?"
answer = answer_from_chunks(question, retrieved_chunks)
print("----- ANSWER -----")
print(answer)

Loading weights: 100%|██████████| 363/363 [00:00<00:00, 59341.79it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----- ANSWER -----

You are a Clinical Data Management assistant.

Answer the question below using ONLY the context provided. Cite PDF name, page, and heading when possible.

Context:
Source: SCDM-Position-Paper.pdf, Page: 5, Heading: Evolution Overview
Text: CDM roles evolve into CDS roles...

Source: 2020_Evolution-of-CDM.pdf, Page: 8, Heading: Skills Needed
Text: Clinical Data Scientists need knowledge in SAS, R...

Question: How should a Clinical Data Manager transition to Data Science roles?

Answer (concise, cite sources):
A Clinical Data Manager should transition to Data Science roles by developing skills in programming languages such as SAS and R, as well as understanding data management and analytics techniques. Additionally, they should learn about clinical trial processes and regulatory requirements to provide a comprehensive understanding of the industry. This will enable them to succeed in Data Science roles within the clinical research field.


In [13]:
from transformers import pipeline

# ----------------------------
# Initialize Nous Hermes model
# ----------------------------
generator = pipeline(
    "text-generation",
    model="NousResearch/Nous-Hermes-13b",
    device=-1,       # -1 for CPU, 0 for GPU
    do_sample=True,
    temperature=0.7
)

# ----------------------------
# Replacement for ask_llm
# ----------------------------
def ask_llm(question, context, max_tokens=300):
    """
    Args:
        question (str): user question
        context (str): combined text from retrieved chunks
        max_tokens (int): max tokens to generate
    Returns:
        str: generated answer
    """
    prompt = f"""
Answer the question using ONLY the context below.
Always mention the source.

Context:
{context}

Question:
{question}
"""

    # Generate output
    output = generator(prompt, max_new_tokens=max_tokens)[0]['generated_text']
    return output

# ----------------------------
# Example usage
# ----------------------------
# Suppose `results` is your list of retrieved chunks
results = [
    {"text": "CDM roles evolve into CDS roles...", "source": "SCDM-Position-Paper.pdf", "page": 5, "heading": "Evolution Overview"},
    {"text": "Clinical Data Scientists need knowledge in SAS, R...", "source": "2020_Evolution-of-CDM.pdf", "page": 8, "heading": "Skills Needed"}
]

context = "\n\n".join([f"Source: {r.get('source','Unknown')}, Page: {r.get('page','Unknown')}, Heading: {r.get('heading','Unknown')}\nText: {r['text']}" for r in results])

question = "How should a Clinical Data Manager transition to Data Science roles?"
answer = ask_llm(question, context)
print("----- ANSWER -----")
print(answer)

Loading weights: 100%|██████████| 363/363 [00:00<00:00, 72601.80it/s]
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----- ANSWER -----

Answer the question using ONLY the context below.
Always mention the source.

Context:
Source: SCDM-Position-Paper.pdf, Page: 5, Heading: Evolution Overview
Text: CDM roles evolve into CDS roles...

Source: 2020_Evolution-of-CDM.pdf, Page: 8, Heading: Skills Needed
Text: Clinical Data Scientists need knowledge in SAS, R...

Question:
How should a Clinical Data Manager transition to Data Science roles?

Answer:
To transition from a Clinical Data Manager (CDM) role to a Data Science role, one should focus on developing skills in data manipulation and analysis using programming languages such as SAS and R. Additionally, gaining experience in data visualization tools and bioinformatics will be beneficial for a successful transition.
